In [40]:
#download
#making frame
#downloading
!pip install pandas
!pip install pandas numpy scikit-learn matplotlib seaborn
!pip install matplotlib.pyplot
!pip install seaborn
!pip install statsmodels
!pip install scorecardpy
!pip install setuptools
!pip install --upgrade setuptools
!pip install flask
!pip install flask pandas numpy scikit-learn matplotlib seaborn openpyxl


ERROR: Could not find a version that satisfies the requirement matplotlib.pyplot (from versions: none)
ERROR: No matching distribution found for matplotlib.pyplot



   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpy

In [38]:
#import libraries 
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.tree import plot_tree

from statsmodels.stats.outliers_influence import variance_inflation_factor

import warnings
warnings.filterwarnings("ignore")

from flask import Flask, render_template, request
import pandas as pd
import numpy as np
import os


from sklearn.metrics import (
    accuracy_score,
    precision_score,
    confusion_matrix
)

import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt

app = Flask(__name__)

UPLOAD_FOLDER = "uploads"

if not os.path.exists(UPLOAD_FOLDER):
    os.makedirs(UPLOAD_FOLDER)

app.config['UPLOAD_FOLDER'] = UPLOAD_FOLDER

uploaded_df = None

In [41]:
#create flaskbackend
from flask import Flask, render_template

app = Flask(__name__)

@app.route('/')
def home():
    return render_template('index.html')

if __name__ == "__main__":
    app.run(debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with stat


SystemExit: 1

In [ ]:
import os
 
import numpy as np
import pandas as pd
from pandas.api.types import is_numeric_dtype
 
from flask import Flask, request, render_template, redirect, url_for
 
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, precision_score
 
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
 
app = Flask(__name__)
 
# Simple in-memory state. Fine for a single-user / single-team internal tool;
# swap for a session or database if multiple people will use this at once.
STATE = {
    "df": None,
    "filename": None,
}
 
 
# -----------------------------
# IDENTIFY VARIABLES
# -----------------------------
def identify_variables(df):
    categorical = []
    numerical = []
    continuous = []
 
    for col in df.columns:
        if is_numeric_dtype(df[col]):
            numerical.append(col)
            if df[col].nunique() > 10:
                continuous.append(col)
        else:
            categorical.append(col)
 
    return categorical, numerical, continuous
 
 
# -----------------------------
# MISSING VALUE TREATMENT
# -----------------------------
def handle_missing(df):
    for col in df.columns:
        if is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(df[col].median())
        else:
            mode_val = df[col].mode()
            if len(mode_val) > 0:
                df[col] = df[col].fillna(mode_val[0])
    return df
 
 
# -----------------------------
# OUTLIER TREATMENT (cap at 2.5th / 97.5th percentile)
# -----------------------------
def outlier_treatment(df):
    for col in df.columns:
        if is_numeric_dtype(df[col]):
            lower = df[col].quantile(0.025)
            upper = df[col].quantile(0.975)
            df[col] = np.where(df[col] < lower, lower, df[col])
            df[col] = np.where(df[col] > upper, upper, df[col])
    return df
 
 
# -----------------------------
# HOME / UPLOAD FORM
# -----------------------------
@app.route('/')
def home():
    return render_template('home.html')
 
 
# -----------------------------
# UPLOAD
# -----------------------------
@app.route('/upload', methods=['POST'])
def upload():
    file = request.files.get('file')
 
    if not file or file.filename == '':
        return render_template('home.html', error="No file was selected. Please choose a CSV file.")
 
    if not file.filename.lower().endswith('.csv'):
        return render_template('home.html', error="Please upload a .csv file.")
 
    try:
        df = pd.read_csv(file)
    except Exception as exc:
        return render_template('home.html', error=f"Could not read that file ({exc}).")
 
    if df.empty or len(df.columns) == 0:
        return render_template('home.html', error="That file looks empty. Check it has a header row and data.")
 
    STATE["df"] = df
    STATE["filename"] = file.filename
 
    categorical, numerical, continuous = identify_variables(df)
 
    return render_template(
        'config.html',
        columns=list(df.columns),
        categorical=categorical,
        numerical=numerical,
        continuous=continuous,
        n_rows=len(df),
        n_cols=len(df.columns),
        filename=file.filename,
    )
 
 
# -----------------------------
# TRAIN
# -----------------------------
@app.route('/train', methods=['POST'])
def train():
    if STATE["df"] is None:
        return redirect(url_for('home'))
 
    df = STATE["df"].copy()
 
    target = request.form['target']
    depth = int(request.form['depth'])
    leaf_nodes = int(request.form['leaf_nodes'])
 
    # Missing value treatment
    df = handle_missing(df)
 
    # Outlier treatment
    df = outlier_treatment(df)
 
    # Separate X & y
    y = df[target]
    X = df.drop(columns=[target])
 
    # One-hot encode categorical predictors
    X = pd.get_dummies(X, drop_first=True)
 
    # Train / test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42
    )
 
    # Decision tree
    model = DecisionTreeClassifier(
        criterion='entropy',
        max_depth=depth,
        max_leaf_nodes=leaf_nodes,
        random_state=42,
    )
    model.fit(X_train, y_train)
 
    # Prediction
    y_pred = model.predict(X_test)
 
    accuracy = round(accuracy_score(y_test, y_pred), 4)
 
    # precision_score needs a positive label / binary target; guard for
    # multiclass targets so this doesn't crash the whole route.
    try:
        precision = round(
            precision_score(y_test, y_pred, average='binary', pos_label=y.unique()[0]),
            4,
        )
    except Exception:
        precision = None
 
    # Plot tree
    plt.figure(figsize=(20, 10))
    plot_tree(
        model,
        feature_names=X_train.columns,
        filled=True,
        rounded=True,
    )
 
    os.makedirs("static", exist_ok=True)
    plt.savefig("static/tree.png", bbox_inches='tight')
    plt.close()
 
    return render_template(
        'result.html',
        accuracy=accuracy,
        precision=precision,
        target=target,
        depth=depth,
        leaf_nodes=leaf_nodes,
        n_test=len(X_test),
    )
 
 
if __name__ == "__main__":
    app.run(debug=False, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
[2026-06-23 11:11:27,806] ERROR in app: Exception on / [GET]
Traceback (most recent call last):
  File "c:\Users\Sailesh\Desktop\amc intern\.venv\Lib\site-packages\flask\app.py", line 1511, in wsgi_app
    response = self.full_dispatch_request()
  File "c:\Users\Sailesh\Desktop\amc intern\.venv\Lib\site-packages\flask\app.py", line 919, in full_dispatch_request
    rv = self.handle_user_exception(e)
  File "c:\Users\Sailesh\Desktop\amc intern\.venv\Lib\site-packages\flask\app.py", line 917, in full_dispatch_request
    rv = self.dispatch_request()
  File "c:\Users\Sailesh\Desktop\amc intern\.venv\Lib\site-packages\flask\app.py", line 902, in dispatch_request
    return self.ensure_sync(self.view_functions[rule.endpoint])(**view_args)  # type: ignore[no-any-return]
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^
  File "C:\Users\Sailesh\AppData\Local\Temp\ipykernel_21236\3074424990.py", line 78, in hom